# L4: Verification Loop

<p style="background-color:#fff6e4; padding:15px; border-width:3px; border-color:#f5ecda; border-style:solid; border-radius:6px"> ⏳ <b>Note <code>(Kernel Starting)</code>:</b> This notebook takes about 30 seconds to be ready to use. You may start and watch the video while you wait.</p>

In [1]:
import warnings
warnings.filterwarnings('ignore')

from IPython.display import display, Markdown, Image

from deepagent_helpers import (DEFAULT_RUBRIC, GRADER_PROMPT,
    langsmith_status, get_ticket,
    build_review_agent, run_verification_loop_live,
    save_verification_trace, load_verification_trace,
)
from visualization_helpers import (
    render_review_trace, render_verification_summary,
    render_grading_grid, render_revision_progression,render_grader_feedback,
    render_report, render_report_progression, render_agent_middleware,
)

print("✓ Environment ready")

✓ Environment ready


<div style="background-color:#fff6ff; padding:13px; border-width:3px; border-color:#efe6ef; border-style:solid; border-radius:6px">
<p> 💻 &nbsp; <b>Access <code>requirements.txt</code> file:</b> 1) click on the <em>"File"</em> option on the top menu of the notebook and then 2) click on <em>"Open"</em>.</p>

<p> ⬇ &nbsp; <b>Download Notebooks:</b> 1) click on the <em>"File"</em> option on the top menu of the notebook and then 2) click on <em>"Download as"</em> and select <em>"Notebook (.ipynb)"</em>.</p>
</div>

## The Rubric


In [2]:
display(Markdown("**The rubric:**\n\n" + DEFAULT_RUBRIC))

**The rubric:**

- completeness: every clause of the requirement document is addressed somewhere in the analysis
- clarity: every bullet in the Ambiguities section has an indented "Resolved by:" line naming the specific fact or decision that would settle it
- depth: every bullet in the Ambiguities section has an indented "Edge case:" line naming a concrete production failure that ambiguity could cause

## Building the Verification Loop

In [3]:
MODEL = "openai:gpt-5.4-mini"
GRADER_MODEL = "openai:gpt-5.4-mini"
MAX_RUBRIC_ITERATIONS = 3

agent = build_review_agent(
    model=MODEL,
    grader_model=GRADER_MODEL,
    rubric_iterations=MAX_RUBRIC_ITERATIONS,   
    max_turns=40,
)
print("✓ Verification loop agent created")

✓ Verification loop agent created


## Run the Agent

In [4]:
TICKET_ID = "NOTIF-205"
requirement_doc = get_ticket(TICKET_ID)

print(f"Ticket {TICKET_ID}:\n")
print(requirement_doc)

Ticket NOTIF-205:

REQUIREMENT: Real-time Notification System

The system shall deliver notifications to users in a timely manner.

Notifications must be sent through appropriate channels based on user preferences.

The system should handle high volumes of notifications efficiently.

Users must be able to customize their notification settings easily.

Critical notifications require immediate delivery.

The system shall track notification delivery status reliably.



<p style="background-color:#f7fff8; padding:15px; border-width:3px; border-color:#e0f0e0; border-style:solid; border-radius:6px"> 🚨
&nbsp; <b>Different Run Results:</b> LLMs do not always produce the same results. Consequently, the agent behavior you experience when running labs may differ from the video.</p>

In [5]:
USE_LIVE_LLM = True
if USE_LIVE_LLM:
    run = run_verification_loop_live(agent, ticket_id=TICKET_ID, 
                                     rubric=DEFAULT_RUBRIC)
    save_verification_trace(run, "lesson_4_live_trace.json")
    print(langsmith_status())
else:
    run = load_verification_trace("lesson_4_prebaked_trace.json")

display(render_verification_summary(run))
display(render_grading_grid(run))

✓ LangSmith tracing is ON — after a live run, go to https://smith.langchain.com, open Tracing in the left-hand menu, and select the project 'AgentLoop'.


**Grader verdicts:** needs_revision → satisfied  
**Turns:** 9 model turns, 16 steps, 2 grading passes  
**Stopped because:** the rubric was satisfied

Criterion,Pass 0,Pass 1
Completeness,Pass,Pass
Clarity,Fail,Pass
Depth,Fail,Pass
Verdict,needs_revision,satisfied
File size (chars),3266,4696


### Execution Trace

Click on the following link to view saved agent execution trace.

🔗   [[ Agent's Trace ]](https://smith.langchain.com/public/9391a9d7-0f2b-4437-adce-924dbeae397e/t?start_time=2026-10-06T08%3A50%3A45.681563Z&run_id=01a11068-f853-7441-a97c-293256873465&conversationTab=trace&trace_id=01a11068-7d31-70f1-b28a-0de12f7bff56&scroll_to=output)

In [6]:
display(render_grader_feedback(run))

## The Final Report

In [7]:
display(render_report_progression(requirement_doc, 
                              run.grading[0].analysis_markdown,
                              run.analysis_markdown))